In [1]:
# =========================
# Cell 1: Import Libraries
# =========================
import cv2
import numpy as np

In [2]:
# =========================
# Cell 2: Camera Setup
# =========================

cap = cv2.VideoCapture(0)

In [3]:
# =========================
# Cell 3: Color Ranges (HSV)
# =========================
colors = {
    "green": (np.array([25, 50, 50]), np.array([95, 255, 255])),
    "red": (np.array([0, 120, 70]), np.array([10, 255, 255])),
    "blue": (np.array([100, 150, 0]), np.array([140, 255, 255]))
}

current_color = "green"

In [4]:
# =========================
# Cell 4: Canvas Setup
# =========================

canvas = None
prev_x, prev_y = 0, 0

In [5]:
def draw_ui(frame):
    # GREEN
    cv2.rectangle(frame, (10, 10), (110, 60), (0, 255, 0), -1)
    cv2.putText(frame, "GREEN", (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,0), 2)

    # RED
    cv2.rectangle(frame, (120, 10), (220, 60), (0, 0, 255), -1)
    cv2.putText(frame, "RED", (140, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 2)

    # BLUE
    cv2.rectangle(frame, (230, 10), (330, 60), (255, 0, 0), -1)
    cv2.putText(frame, "BLUE", (250, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 2)

    # CLEAR
    cv2.rectangle(frame, (340, 10), (440, 60), (200, 200, 200), -1)
    cv2.putText(frame, "CLEAR", (355, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,0), 2)

    return frame

In [6]:
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)

    if canvas is None:
        canvas = np.zeros_like(frame)

    # Draw UI
    frame = draw_ui(frame)

    # Convert to HSV
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    lower, upper = colors[current_color]

    mask = cv2.inRange(hsv, lower, upper)

    # Clean noise
    mask = cv2.erode(mask, None, iterations=2)
    mask = cv2.dilate(mask, None, iterations=2)

    # Show debug mask (optional)
    cv2.imshow("Mask", mask)

    # Find contours
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours:
        c = max(contours, key=cv2.contourArea)

        if cv2.contourArea(c) > 1000:
            x, y, w, h = cv2.boundingRect(c)
            cx, cy = x + w//2, y + h//2

            if prev_x == 0 and prev_y == 0:
                prev_x, prev_y = cx, cy

            cv2.line(canvas, (prev_x, prev_y), (cx, cy), (0,255,0), 5)

            prev_x, prev_y = cx, cy

            cv2.circle(frame, (cx, cy), 8, (0,0,255), -1)
    else:
        prev_x, prev_y = 0, 0

    # Merge
    output = cv2.add(frame, canvas)

    cv2.imshow("AR Drawing Pad", output)

    key = cv2.waitKey(1)

    # ================= UI Controls =================
    if key == ord('g'):
        current_color = "green"

    elif key == ord('r'):
        current_color = "red"

    elif key == ord('b'):
        current_color = "blue"

    elif key == ord('c'):
        canvas = np.zeros_like(frame)

    elif key == 27:  # ESC
        break

In [7]:
# =========================
# Cell 7: Cleanup
# =========================

cap.release()
cv2.destroyAllWindows()